# Portfolio Allocation with a Carbon Objective
## Sustainability Aware Asset Management — Final Notebook

**Region:** Pacific (PAC) | **Scope:** Scope 1 CO₂ Emissions  
**Period:** 2014–2025 (out-of-sample) | **Rebalancing:** Annual (Dec 2013 – Dec 2024)

This notebook reproduces all results for the SAAM project:
- **Part I:** Minimum-variance and value-weighted benchmark portfolios
- **Part II:** Carbon-constrained portfolios (50% footprint cap, tracking-error minimisation, net-zero trajectory)

All cells run top-to-bottom without manual intervention. Data files are expected in a `data/` subdirectory.

## 1. Setup & Configuration

Import all required libraries and set project parameters. Key choices:
- **Estimation window:** 10 years (120 months) of trailing data
- **Minimum observations:** 36 months (per project suggestion of ≥3 years)
- **Stale price threshold:** 50% zero-return months triggers exclusion
- **Low price threshold:** RI values below 0.5 treated as missing

In [31]:
import os
import re
import warnings
from datetime import datetime

try:
    import cvxpy as cp
except ImportError:
    cp = None
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage
from scipy.optimize import minimize

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
REGION_CODE         = "PAC"
LOW_PRICE_THRESHOLD = 0.5
WINDOW_YEARS        = 10
MIN_OBS_MONTHS      = 36
STALE_THRESHOLD     = 0.50
START_YEAR_OOS      = 2014
END_YEAR_OOS        = 2025
DECISION_YEARS      = list(range(2013, 2025))
Y0                  = 2013
THETA               = 0.10
RIDGE               = 1e-8
SLSQP_MAXITER       = 400
SLSQP_FTOL          = 1e-9
USE_LEDOIT_WOLF     = True

RESULTS_PART1       = "resultsPart1"
RESULTS_PART2       = "ResultsPart2_FINAL"

CO2_FILE            = "DS_CO2_SCOPE_1_Y_2025.xlsx"
RF_FILE             = "Risk_Free_Rate_2025.xlsx"

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")

os.makedirs(RESULTS_PART1, exist_ok=True)
os.makedirs(RESULTS_PART2, exist_ok=True)


## 2. Helper Functions — Data Loading & Cleaning

These utilities handle Datastream-formatted Excel files, parse monthly dates,
detect delisting events, forward-fill interior missing values, and apply the
low-price / stale-price filters described in §1 of the project brief.

In [32]:


def data_path(filename: str) -> str:
    from pathlib import Path

    candidates = []

    # Start from common anchors.
    for anchor in [Path(DATA_DIR), Path(BASE_DIR), Path.cwd()]:
        candidates.append(anchor / filename)
        candidates.append(anchor / "data" / filename)

        # Walk up parent folders to handle arbitrary notebook launch paths.
        for parent in [anchor, *anchor.parents]:
            candidates.append(parent / filename)
            candidates.append(parent / "data" / filename)

    # De-duplicate while preserving order.
    seen = set()
    for p in candidates:
        key = str(p.resolve()) if p.exists() else str(p)
        if key in seen:
            continue
        seen.add(key)
        if p.exists():
            return str(p)

    raise FileNotFoundError(f"Could not find required file: {filename}")


# ─────────────────────────────────────────────────────────────────────────────
# SHARED HELPERS — loading & cleaning
# ─────────────────────────────────────────────────────────────────────────────

def load_datastream_wide(filepath, sheet=0):
    df = pd.read_excel(filepath, sheet_name=sheet, header=0, engine="openpyxl")
    df.columns = ["NAME", "ISIN"] + list(df.columns[2:])
    df = df[~df["NAME"].astype(str).str.startswith("$$ER", na=False)]
    df = df.dropna(subset=["ISIN"])
    df["ISIN"] = df["ISIN"].astype(str).str.strip()
    df["NAME"] = df["NAME"].astype(str).str.strip()
    df = df[df["ISIN"] != ""].set_index("ISIN")
    return df


def load_annual_panel(filepath: str) -> pd.DataFrame:
    df = pd.read_excel(filepath, engine="openpyxl")
    df.columns = ["NAME", "ISIN"] + list(df.columns[2:])
    df = df[~df["NAME"].astype(str).str.startswith("$$ER", na=False)]
    df = df.dropna(subset=["ISIN"])
    df["ISIN"] = df["ISIN"].astype(str).str.strip()
    df = df[df["ISIN"] != ""].set_index("ISIN").drop(columns=["NAME"], errors="ignore")
    df.columns = df.columns.astype(int)
    df = df.apply(pd.to_numeric, errors="coerce")
    return df


def load_rf_monthly(filepath):
    df = pd.read_excel(filepath, engine="openpyxl", index_col=0)
    df.index = df.index.astype(str).str.strip()
    dates = pd.to_datetime(df.index, format="%Y%m") + pd.offsets.MonthEnd(0)
    rf = pd.to_numeric(df.iloc[:, 0], errors="coerce") / 100.0
    rf.index = dates
    return rf.dropna().sort_index()


def parse_monthly_columns(cols):
    parsed, keep = [], []
    for c in cols:
        if c == "NAME":
            continue
        dt = pd.to_datetime(str(c), errors="coerce", dayfirst=True)
        if pd.notna(dt):
            keep.append(c)
            parsed.append(pd.Timestamp(dt).normalize())
    return keep, parsed


def extract_delist_date(name_str):
    m = re.search(r"(?:DELIST|DEAD)\.(\d{2}/\d{2}/\d{2,4})", str(name_str))
    if not m:
        return None
    s = m.group(1)
    for fmt in ("%d/%m/%y", "%d/%m/%Y"):
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            pass
    return None


def forward_fill_middle_only(price_df, dates):
    arr = price_df.to_numpy(dtype=float, copy=True)
    for i in range(arr.shape[0]):
        row = arr[i, :]
        valid_idx = np.where(~np.isnan(row))[0]
        if valid_idx.size == 0:
            continue
        first, last = valid_idx[0], valid_idx[-1]
        last_val = row[first]
        for k in range(first + 1, last + 1):
            if np.isnan(row[k]):
                row[k] = last_val
            else:
                last_val = row[k]
        arr[i, :] = row
    return pd.DataFrame(arr, index=price_df.index, columns=dates)


def clean_monthly_ri_prices(raw_ri_m, dates, low_price_threshold=0.5,
                            preserve_december_missing=True):
    prices = raw_ri_m.copy().apply(pd.to_numeric, errors="coerce")
    prices = prices.mask(prices < low_price_threshold)
    dec_cols = [d for d in dates if d.month == 12]
    orig_dec_missing = prices[dec_cols].isna() if preserve_december_missing else None
    prices_filled = forward_fill_middle_only(prices, dates)
    if preserve_december_missing and len(dec_cols) > 0:
        prices_filled.loc[:, dec_cols] = prices_filled.loc[:, dec_cols].mask(orig_dec_missing)
    return prices_filled


def apply_delisting_to_returns(returns_df, price_df, delist_dates, dates):
    out = returns_df.copy()
    date_index = pd.Index(dates)
    for isin, ddate in delist_dates.items():
        if ddate is None or isin not in out.index or isin not in price_df.index:
            continue
        dts = pd.Timestamp(ddate).normalize()
        price_series = price_df.loc[isin]
        on_after_delist = date_index[date_index >= dts]
        if len(on_after_delist) == 0:
            continue
        default_dcol = on_after_delist.min()
        future_window = date_index[date_index >= default_dcol]
        future_prices = price_series.loc[future_window]
        if future_prices.notna().any():
            if not (pd.isna(price_series.loc[default_dcol]) and future_prices.isna().all()):
                continue
        prior_valid = date_index[(date_index < default_dcol) & price_series.loc[date_index].notna()]
        if len(prior_valid) > 0:
            last_valid = prior_valid.max()
            candidate_window = date_index[(date_index > last_valid) & (date_index <= default_dcol)]
            missing = candidate_window[price_series.loc[candidate_window].isna()]
            dcol = missing.min() if len(missing) > 0 else default_dcol
        else:
            dcol = default_dcol
        out.at[isin, dcol] = -1.0
        after = date_index[date_index > dcol]
        if len(after) > 0:
            out.loc[isin, after] = np.nan
    return out


def apply_terminal_loss_for_permanent_disappearances(returns_df, price_df, dates):
    out = returns_df.copy()
    date_index = pd.Index(dates)
    for isin in out.index:
        if isin not in price_df.index:
            continue
        price_series = price_df.loc[isin]
        valid_dates = date_index[price_series.loc[date_index].notna()]
        if len(valid_dates) == 0:
            continue
        last_valid_price_date = valid_dates.max()
        tail_dates = date_index[date_index > last_valid_price_date]
        if len(tail_dates) == 0:
            continue
        if price_series.loc[tail_dates].isna().all():
            first_missing = tail_dates.min()
            if pd.isna(out.at[isin, first_missing]):
                out.at[isin, first_missing] = -1.0
            later = date_index[date_index > first_missing]
            if len(later) > 0:
                out.loc[isin, later] = np.nan
    return out


def adjust_mv_caps_for_terminal_events(mv_caps, ri_returns, dates):
    caps = mv_caps.copy()
    date_index = pd.Index(dates)
    for isin in caps.index.intersection(ri_returns.index):
        series = ri_returns.loc[isin, date_index]
        minus100_dates = date_index[series == -1.0]
        if len(minus100_dates) == 0:
            continue
        for d in minus100_dates:
            after = date_index[date_index > d]
            if len(after) == 0 or series.loc[after].isna().all():
                caps.loc[isin, date_index[date_index >= d]] = np.nan
                break
    return caps


# ─────────────────────────────────────────────────────────────────────────────


## 3. Helper Functions — Investment Set, Estimation & Optimisation

**Investment set:** For each decision year Y, we select firms in the assigned region
with sufficient return history and available carbon data.

**Covariance estimation:** We use the **pairwise sample covariance** approach — for each
pair of assets (i, j), the covariance is estimated using only the months where both
returns are observed. This avoids biasing entries for assets with different coverage periods.

**Optimisation:** Minimum-variance portfolios are solved via SLSQP; carbon-constrained
portfolios use CVXPY (OSQP/SCS) when available, with SLSQP as fallback.

In [33]:
# SHARED HELPERS — investment set, moments, optimisation, performance
# ─────────────────────────────────────────────────────────────────────────────

def year_end_col(dates, year):
    decs = [d for d in dates if d.year == year and d.month == 12]
    return max(decs) if decs else None


def window_cols(dates, year_end, window_years=10):
    start = pd.Timestamp(f"{year_end - window_years + 1}-01-01")
    end = pd.Timestamp(f"{year_end}-12-31")
    return [d for d in dates if start <= d <= end]


def stale_mask(returns_df, cols, threshold=0.50):
    sub = returns_df[cols]
    denom = sub.notna().sum(axis=1).replace(0, np.nan)
    frac0 = (sub == 0).sum(axis=1) / denom
    return frac0 > threshold


def build_investment_set(ri_prices, ri_returns, dates, year_end, co2_panel=None):
    dec_col = year_end_col(dates, year_end)
    if dec_col is None:
        raise ValueError(f"No December column for year {year_end}.")
    cols = window_cols(dates, year_end, WINDOW_YEARS)
    ok_price = ri_prices[dec_col].notna()
    n_obs = ri_returns[cols].notna().sum(axis=1)
    ok_obs = n_obs >= MIN_OBS_MONTHS
    ok_stale = ~stale_mask(ri_returns, cols, STALE_THRESHOLD).fillna(True)
    ok_carbon = pd.Series(True, index=ri_prices.index)
    if co2_panel is not None:
        if year_end in co2_panel.columns:
            ok_carbon = co2_panel[year_end].reindex(ri_prices.index).notna().fillna(False)
        else:
            print(f"Warning: year {year_end} not in CO2 panel — carbon filter skipped.")
    eligible = ri_prices.index[ok_price & ok_obs & ok_stale & ok_carbon]
    return list(eligible), cols

# Pairwise covariance estimation following the project brief (§2.1):
# For each pair (i,j), we use only the months where both returns are observed.
# This avoids biasing off-diagonal entries when coverage differs across assets.
def estimate_moments(ri_returns, isins, cols, ridge=1e-8):
    R = ri_returns.loc[isins, cols].astype(float)
    N = len(isins)

    # Asset-level means (each computed from its own valid observations)
    mu = R.mean(axis=1, skipna=True).to_numpy()

    # Pairwise covariance matrix
    R_np = R.to_numpy()  # shape (N, T)
    Sigma = np.zeros((N, N))
    for i in range(N):
        ri = R_np[i, :]
        valid_i = np.isfinite(ri)
        for j in range(i, N):
            rj = R_np[j, :]
            overlap = valid_i & np.isfinite(rj)
            n_overlap = overlap.sum()
            if n_overlap < 2:
                Sigma[i, j] = Sigma[j, i] = 0.0
                continue
            ri_ov = ri[overlap]
            rj_ov = rj[overlap]
            cov_ij = np.mean((ri_ov - ri_ov.mean()) * (rj_ov - rj_ov.mean()))
            Sigma[i, j] = cov_ij
            Sigma[j, i] = cov_ij

    if np.isnan(Sigma).any():
        raise ValueError("Covariance matrix contains NaN after pairwise estimation.")
    Sigma = Sigma + ridge * np.eye(N)
    return mu, Sigma


def solve_min_variance(Sigma):
    n = Sigma.shape[0]
    a0 = np.full(n, 1.0 / n)
    res = minimize(
        lambda a: float(a @ Sigma @ a),
        a0,
        method="SLSQP",
        bounds=[(0.0, 1.0)] * n,
        constraints=[{"type": "eq", "fun": lambda a: np.sum(a) - 1.0}],
        options={"maxiter": SLSQP_MAXITER, "ftol": SLSQP_FTOL, "disp": False},
    )
    if not res.success:
        raise RuntimeError(f"SLSQP failed: {res.message}")
    w = np.where(res.x < 1e-10, 0.0, res.x)
    return w / w.sum()

# Tracking-error and carbon-constrained portfolios solved via CVXPY (preferred)
# or SLSQP fallback. Objective: min (w-w_ref)' Σ (w-w_ref) for TE mode,
# or min w' Σ w for MV mode, subject to carbon + weight constraints.
def solve_cvxpy(Sigma_arr, e_c, cf_target, w_ref=None, mode="mv"):
    N = Sigma_arr.shape[0]
    Sigma_arr = 0.5 * (Sigma_arr + Sigma_arr.T)
    eigvals, eigvecs = np.linalg.eigh(Sigma_arr)
    min_eig = float(np.min(eigvals))
    if min_eig < 1e-10:
        eigvals = np.maximum(eigvals, 1e-10)
        Sigma_arr = eigvecs @ np.diag(eigvals) @ eigvecs.T
        Sigma_arr = 0.5 * (Sigma_arr + Sigma_arr.T)

    def _is_feasible(w):
        if w is None:
            return False
        if not np.all(np.isfinite(w)):
            return False
        if np.any(w < -1e-8):
            return False
        if abs(float(np.sum(w)) - 1.0) > 1e-6:
            return False
        if float(w @ e_c) > float(cf_target) + 1e-6:
            return False
        return True
        
    def _solve_slsqp():
        x0 = w_ref.copy() if (mode == "te" and w_ref is not None) else np.full(N, 1.0 / N)
        def objective(a):
            if mode == "mv":
                diff = a
            else:
                diff = a - w_ref
            return float(diff @ Sigma_arr @ diff)
        constraints = [
            {"type": "eq", "fun": lambda a: np.sum(a) - 1.0},
            {"type": "ineq", "fun": lambda a: cf_target - float(a @ e_c)},
        ]
        res = minimize(
            objective,
            x0,
            method="SLSQP",
            bounds=[(0.0, 1.0)] * N,
            constraints=constraints,
            options={"maxiter": 1000, "ftol": 1e-9, "disp": False},
        )
        if not res.success:
            return None
        w = np.maximum(res.x, 0.0)
        s = w.sum()
        if s <= 0:
            return None
        w = w / s
        return w if _is_feasible(w) else None

    if cp is None:
        return _solve_slsqp()

    Sp = cp.psd_wrap(Sigma_arr)
    alpha = cp.Variable(N)
    if mode == "mv":
        obj = cp.Minimize(cp.quad_form(alpha, Sp))
    else:
        obj = cp.Minimize(cp.quad_form(alpha - w_ref, Sp))
    prob = cp.Problem(obj, [cp.sum(alpha) == 1, alpha >= 0, alpha @ e_c <= cf_target])
    for solver in (cp.OSQP, cp.SCS):
        try:
            kw = {"eps_abs": 1e-8, "eps_rel": 1e-8, "max_iter": 20_000} if solver == cp.OSQP else {}
            prob.solve(solver=solver, **kw)
        except Exception:
            continue
        if alpha.value is not None and prob.status in ("optimal", "optimal_inaccurate"):
            w = np.maximum(alpha.value, 0.0)
            s = w.sum()
            if s <= 0:
                continue
            w = w / s
            if _is_feasible(w):
                return w
    return _solve_slsqp()

# OOS return calculation with weight drift (buy-and-hold within each year).
# Raises an error if a position has missing returns, rather than silently filling.
def compute_mv_oos_returns(portfolios, ri_returns, dates):
    out_r, out_d = [], []
    WEIGHT_TOL = 1e-12
    for year_end, info in portfolios.items():
        invest_year = year_end + 1
        year_months = [d for d in dates if pd.Timestamp(f"{invest_year}-01-01") <= d <= pd.Timestamp(f"{invest_year}-12-31")]
        if not year_months:
            continue
        isins = info["isins"]
        w = info["weights"].copy()
        for d in year_months:
            r_series = ri_returns.loc[isins, d].copy()
            missing_mask = r_series.isna().to_numpy()
            bad_mask = missing_mask & (w > WEIGHT_TOL)
            if bad_mask.any():
                bad_isins = list(np.array(isins)[bad_mask])
                bad_weights = w[bad_mask]
                raise ValueError(
                    f"Missing realized returns for invested names on {d.date()}. ISINs={bad_isins}, weights={bad_weights.tolist()}"
                )
            r_i = r_series.fillna(0.0).to_numpy()
            r_p = float(w @ r_i)
            out_r.append(r_p)
            out_d.append(d)
            denom = 1.0 + r_p
            if denom <= 0:
                raise ValueError(f"Portfolio value collapsed to non-positive level on {d.date()} (1 + r_p = {denom}).")
            w = w * (1.0 + r_i) / denom
            w = np.where(w < WEIGHT_TOL, 0.0, w)
            if w.sum() <= 0:
                raise ValueError(f"Portfolio weights sum to zero after update on {d.date()}.")
            w = w / w.sum()
    return pd.Series(out_r, index=pd.to_datetime(out_d)).sort_index()


def compute_vw_oos_returns(portfolios, ri_returns, mv_caps, dates, start_year=2014, end_year=2025):
    out_r, out_d = [], []
    dates_sorted = sorted(dates)
    pos = {d: i for i, d in enumerate(dates_sorted)}
    for year_end, info in portfolios.items():
        invest_year = year_end + 1
        if invest_year < start_year or invest_year > end_year:
            continue
        eligible_isins = list(info["isins"])
        year_months = [d for d in dates_sorted if pd.Timestamp(f"{invest_year}-01-01") <= d <= pd.Timestamp(f"{invest_year}-12-31")]
        for d in year_months:
            i = pos.get(d)
            if i is None or i == 0:
                continue
            d_prev = dates_sorted[i - 1]
            if d_prev not in mv_caps.columns:
                raise ValueError(f"Lagged market-cap column missing for benchmark month {d_prev.date()}")
            if d not in ri_returns.columns:
                raise ValueError(f"Return column missing for benchmark month {d.date()}")
            caps = mv_caps.reindex(eligible_isins)[d_prev]
            rets = ri_returns.reindex(eligible_isins)[d]
            caps_missing = caps.isna()
            rets_missing = rets.isna()
            inconsistent = caps_missing ^ rets_missing
            if inconsistent.any():
                bad_isins = list(caps.index[inconsistent])
                raise ValueError(
                    f"VW benchmark inconsistent data on {d.date()}.\nDropped ISINs: {bad_isins}\nReason: market cap and return availability do not match."
                )
            valid = (~caps_missing) & (~rets_missing)
            if valid.sum() == 0:
                raise ValueError(f"No valid benchmark constituents on {d.date()}")
            cap_sum = caps.loc[valid].sum()
            if pd.isna(cap_sum) or cap_sum <= 0:
                raise ValueError(f"Invalid lagged market-cap sum for benchmark on {d.date()}")
            w = caps.loc[valid] / cap_sum
            r_p = float(w.to_numpy() @ rets.loc[valid].to_numpy())
            out_r.append(r_p)
            out_d.append(d)
    return pd.Series(out_r, index=pd.to_datetime(out_d)).sort_index()


def compute_oos_returns(portfolios_dict, ri_ret, dates, start_yr, end_yr):
    out_r, out_d = [], []
    WEIGHT_TOL = 1e-12

    for year_end in sorted(portfolios_dict.keys()):
        invest_year = year_end + 1
        if invest_year < start_yr or invest_year > end_yr:
            continue

        year_months = [d for d in dates if d.year == invest_year]
        if not year_months:
            continue

        isins = portfolios_dict[year_end]["isins"]
        w = portfolios_dict[year_end]["weights"].copy()

        for d in year_months:
            r_series = ri_ret.loc[isins, d].copy()

            missing_mask = r_series.isna().to_numpy()
            bad_mask = missing_mask & (w > WEIGHT_TOL)

            if bad_mask.any():
                bad_isins = list(np.array(isins)[bad_mask])
                bad_weights = w[bad_mask]
                raise ValueError(
                    f"Missing realized returns for invested names on {d.date()}. "
                    f"ISINs={bad_isins}, weights={bad_weights.tolist()}"
                )

            r_i = r_series.fillna(0.0).to_numpy()

            r_p = float(w @ r_i)
            out_r.append(r_p)
            out_d.append(d)

            denom = 1.0 + r_p
            if denom <= 0:
                raise ValueError(
                    f"Portfolio value collapsed to non-positive level on {d.date()} "
                    f"(1 + r_p = {denom})."
                )

            w = w * (1.0 + r_i) / denom
            w = np.where(w < WEIGHT_TOL, 0.0, w)

            if w.sum() <= 0:
                raise ValueError(f"Portfolio weights sum to zero after update on {d.date()}.")

            w = w / w.sum()

    return pd.Series(out_r, index=pd.to_datetime(out_d)).sort_index()


def validate_oos_series(series: pd.Series, name: str):
    expected_months = pd.date_range(start=f"{START_YEAR_OOS}-01-31", end=f"{END_YEAR_OOS}-12-31", freq="M")
    s = series.copy()
    s.index = pd.to_datetime(s.index) + pd.offsets.MonthEnd(0)
    s = s[~s.index.duplicated(keep="last")].reindex(expected_months)
    if s.isna().any():
        missing = [d.strftime("%Y-%m-%d") for d in s.index[s.isna()]]
        raise ValueError(f"{name} is missing OOS returns for months: {missing}")
    if len(s) != 144:
        raise ValueError(f"{name} has {len(s)} observations, expected 144.")
    return s


def perf_stats(r, rf=None):
    r = r.dropna()
    T = len(r)
    if T == 0:
        return {
            "Annualized Average Return": np.nan,
            "Annualized Volatility": np.nan,
            "Annualized Cumulative Return": np.nan,
            "Sharpe Ratio": np.nan,
            "Minimum Monthly Return": np.nan,
            "Maximum Monthly Return": np.nan,
        }
    rf_aligned = rf.reindex(r.index).fillna(0.0) if rf is not None else pd.Series(0.0, index=r.index)
    excess = r - rf_aligned
    mean_m = r.mean()
    vol_m = r.std(ddof=1)
    vol_excess = excess.std(ddof=1)
    return {
        "Annualized Average Return": 12.0 * mean_m,
        "Annualized Volatility": vol_m * np.sqrt(12.0),
        "Annualized Cumulative Return": (1.0 + r).prod() ** (12.0 / T) - 1.0,
        "Sharpe Ratio": (excess.mean() / vol_excess) * np.sqrt(12.0) if vol_excess > 0 else np.nan,
        "Minimum Monthly Return": r.min(),
        "Maximum Monthly Return": r.max(),
    }


def perf_stats_extended(r: pd.Series, rf: pd.Series = None) -> dict:
    r = r.dropna()
    T = len(r)
    keys = [
        "Annualized Average Return", "Annualized Volatility",
        "Annualized Cumulative Return", "Sharpe Ratio",
        "Minimum Monthly Return", "Maximum Monthly Return",
        "VaR_95 (monthly)", "VaR_99 (monthly)",
        "ES_95 (monthly)", "ES_99 (monthly)", "Max_Drawdown",
    ]
    if T == 0:
        return {k: np.nan for k in keys}
    rf_aligned = rf.reindex(r.index).fillna(0.0) if rf is not None else pd.Series(0.0, index=r.index)
    excess = r - rf_aligned
    mean_m = r.mean()
    vol_m = r.std(ddof=1)
    vol_exc = excess.std(ddof=1)
    var95 = -float(np.percentile(r, 5))
    var99 = -float(np.percentile(r, 1))
    tail95 = r[r <= -var95]
    es95 = float(-tail95.mean()) if len(tail95) > 0 else var95
    tail99 = r[r <= -var99]
    es99 = float(-tail99.mean()) if len(tail99) > 0 else var99
    cum = (1.0 + r).cumprod()
    running_max = cum.cummax()
    mdd = float((cum / running_max - 1.0).min())
    return {
        "Annualized Average Return": 12.0 * mean_m,
        "Annualized Volatility": vol_m * np.sqrt(12.0),
        "Annualized Cumulative Return": (1.0 + r).prod() ** (12.0 / T) - 1.0,
        "Sharpe Ratio": (excess.mean() / vol_exc * np.sqrt(12.0) if vol_exc > 0 else np.nan),
        "Minimum Monthly Return": r.min(),
        "Maximum Monthly Return": r.max(),
        "VaR_95 (monthly)": var95,
        "VaR_99 (monthly)": var99,
        "ES_95 (monthly)": es95,
        "ES_99 (monthly)": es99,
        "Max_Drawdown": mdd,
    }


def perf_stats_relative(r_portfolio: pd.Series, r_benchmark: pd.Series, portfolio_weights: dict, benchmark_weights: dict, rf: pd.Series = None) -> dict:
    r_p = r_portfolio.dropna()
    r_b = r_benchmark.reindex(r_p.index).dropna()
    common = r_p.index.intersection(r_b.index)
    r_p, r_b = r_p[common], r_b[common]
    active = r_p - r_b
    ann_active = 12.0 * active.mean()
    te_ann = active.std(ddof=1) * np.sqrt(12.0)
    ir = ann_active / te_ann if te_ann > 0 else np.nan
    active_shares = []
    for Y in sorted(set(portfolio_weights) & set(benchmark_weights)):
        p_info = portfolio_weights[Y]
        b_info = benchmark_weights[Y]
        all_isins = list(set(p_info["isins"]) | set(b_info["isins"]))
        wp = pd.Series(dict(zip(p_info["isins"], p_info["weights"]))).reindex(all_isins).fillna(0.0)
        wb = pd.Series(dict(zip(b_info["isins"], b_info["weights"]))).reindex(all_isins).fillna(0.0)
        active_shares.append(0.5 * np.abs(wp.values - wb.values).sum())
    return {
        "Active Return (ann.)": ann_active,
        "Tracking Error (ann.)": te_ann,
        "Information Ratio": ir,
        "Avg Active Share": float(np.mean(active_shares)) if active_shares else np.nan,
    }


def export_part1_excel_template(template_path, output_path, stats, out_df):
    wb = load_workbook(template_path)
    ws = wb["Sheet1"]
    ws["B3"] = float(stats.loc["Value Weighted", "Annualized Average Return"])
    ws["B4"] = float(stats.loc["Value Weighted", "Annualized Volatility"])
    ws["B5"] = float(stats.loc["Value Weighted", "Annualized Cumulative Return"])
    ws["B6"] = float(stats.loc["Value Weighted", "Sharpe Ratio"])
    ws["B7"] = float(stats.loc["Value Weighted", "Minimum Monthly Return"])
    ws["B8"] = float(stats.loc["Value Weighted", "Maximum Monthly Return"])
    ws["C3"] = float(stats.loc["Minimum Variance", "Annualized Average Return"])
    ws["C4"] = float(stats.loc["Minimum Variance", "Annualized Volatility"])
    ws["C5"] = float(stats.loc["Minimum Variance", "Annualized Cumulative Return"])
    ws["C6"] = float(stats.loc["Minimum Variance", "Sharpe Ratio"])
    ws["C7"] = float(stats.loc["Minimum Variance", "Minimum Monthly Return"])
    ws["C8"] = float(stats.loc["Minimum Variance", "Maximum Monthly Return"])
    if len(out_df) != 144:
        raise ValueError(f"Expected 144 monthly observations, got {len(out_df)}")
    start_row = 3
    for i, row in enumerate(out_df.itertuples(index=False), start=start_row):
        ws[f"F{i}"] = float(row.VW_Return)
        ws[f"G{i}"] = float(row.MV_Return)
    plot_path = os.path.join(os.path.dirname(output_path), "part1_cumulative_plot.png")
    plt.figure(figsize=(9, 4.8))
    plt.plot(out_df["Date"], out_df["VW_CumReturn"], label="Value-weighted portfolio")
    plt.plot(out_df["Date"], out_df["MV_CumReturn"], label="Minimum-variance portfolio")
    plt.title("Cumulative Returns (2014–2025)")
    plt.xlabel("Date")
    plt.ylabel("Growth of $1")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(plot_path, dpi=200, bbox_inches="tight")
    plt.close()
    img = XLImage(plot_path)
    img.width = 520
    img.height = 280
    ws.add_image(img, "B10")
    wb.save(output_path)


# ─────────────────────────────────────────────────────────────────────────────
# SHARED HELPERS — carbon metrics, plots, shrinkage
# ─────────────────────────────────────────────────────────────────────────────

def carbon_intensity_vec(e, r):
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(r > 0, e / r, np.nan)

# WACI: re-normalize weights over firms with valid CI (revenue > 0).
def waci_metric(w, ci):
    valid = ~np.isnan(ci)
    if not valid.any():
        return np.nan
    w2 = np.where(valid, w, 0.0)
    s = w2.sum()
    return float((w2 / s) @ np.where(valid, ci, 0.0)) if s > 0 else np.nan


def cf_metric(w, e, c):
    with np.errstate(invalid="ignore", divide="ignore"):
        ratio = np.where(c > 0, e / c, 0.0)
    return float(w @ ratio)


def cf_vw_metric(e, c):
    total = c.sum()
    return float(e.sum() / total) if total > 0 else np.nan


def e_over_c_vec(e, c):
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(c > 0, e / c, 0.0)


CF_TOL = 1e-4

def validate_cf_constraint(w, e, c, cf_target, label, Y):
    achieved = cf_metric(w, e, c)
    if not np.isfinite(achieved):
        raise ValueError(f"{label} | Y={Y}: achieved CF is not finite.")
    if achieved > cf_target + CF_TOL:
        raise ValueError(f"{label} | Y={Y}: CF constraint violated. Achieved={achieved:.8f}, Target={cf_target:.8f}")
    return achieved


def save_cumret_plot(series_dict, filename, title):
    plt.figure(figsize=(10, 6))
    for label, s in series_dict.items():
        s = s.dropna().sort_index()
        wealth = (1.0 + s).cumprod()
        plt.plot(wealth.index, wealth.values, label=label)
    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Cumulative wealth")
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=200)
    plt.close()


def save_annual_line_plot(df, filename, title, ylabel):
    plt.figure(figsize=(10, 6))
    for col in df.columns:
        plt.plot(df.index, df[col], marker="o", label=col)
    plt.title(title)
    plt.xlabel("Year")
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=200)
    plt.close()


def plot_cumret_drawdown(series_dict, filename, title, figsize=(12, 7)):
    """Cumulative return (top) + drawdown (bottom) dual-panel plot."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, height_ratios=[3, 1],
                                    sharex=True, gridspec_kw={"hspace": 0.08})
    colors = plt.cm.Set1.colors
    for idx, (label, s) in enumerate(series_dict.items()):
        s = s.dropna().sort_index()
        wealth = (1.0 + s).cumprod()
        running_max = wealth.cummax()
        dd = wealth / running_max - 1.0
        c = colors[idx % len(colors)]
        ax1.plot(wealth.index, wealth.values, label=label, color=c, linewidth=1.2)
        ax2.fill_between(dd.index, dd.values, 0, alpha=0.3, color=c)
        ax2.plot(dd.index, dd.values, color=c, linewidth=0.8)
        # annotate final value
        ax1.annotate(f"{wealth.iloc[-1]:.2f}x", xy=(wealth.index[-1], wealth.iloc[-1]),
                     fontsize=8, color=c, ha="left")
    ax1.set_title(title, fontsize=12)
    ax1.set_ylabel("Growth of $1")
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.2)
    ax2.set_ylabel("Drawdown")
    ax2.set_xlabel("Date")
    ax2.grid(True, alpha=0.2)
    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches="tight")
    plt.close(fig)


def plot_cf_bar_comparison(cf_dict, filename, title, constraint_dict=None, figsize=(13, 6)):
    """Grouped bar chart for carbon footprint comparison across portfolios/years."""
    df = pd.DataFrame(cf_dict).sort_index()
    years = df.index.values
    n_series = len(df.columns)
    width = 0.8 / n_series
    colors = plt.cm.Set2.colors

    fig, ax = plt.subplots(figsize=figsize)
    for idx, col in enumerate(df.columns):
        x = np.arange(len(years)) + idx * width
        bars = ax.bar(x, df[col].values, width=width, label=col, color=colors[idx % len(colors)],
                      edgecolor="white", linewidth=0.5)
        # value labels on bars
        for bar, val in zip(bars, df[col].values):
            if np.isfinite(val) and val > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                        f"{val:.0f}", ha="center", va="bottom", fontsize=6)

    if constraint_dict:
        for label, vals in constraint_dict.items():
            s = pd.Series(vals).reindex(years)
            ax.plot(np.arange(len(years)) + (n_series - 1) * width / 2, s.values,
                    "--", linewidth=1.5, label=label)

    ax.set_xticks(np.arange(len(years)) + (n_series - 1) * width / 2)
    ax.set_xticklabels(years, fontsize=9)
    ax.set_title(title, fontsize=12)
    ax.set_ylabel("tCO2-eq / M USD")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches="tight")
    plt.close(fig)


def plot_correlation_heatmap(corr_matrix, labels, filename, title="Return Correlation Matrix"):
    """Heatmap of pairwise return correlations."""
    fig, ax = plt.subplots(figsize=(7, 5.5))
    im = ax.imshow(corr_matrix, cmap="RdYlGn", vmin=0.5, vmax=1.0)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=8, rotation=45, ha="right")
    ax.set_yticklabels(labels, fontsize=8)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{corr_matrix[i, j]:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title, fontsize=12)
    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches="tight")
    plt.close(fig)

def export_weight_comparison(reference_portfolios, candidate_portfolios, output_csv):
    """
    Compare two sets of annual portfolio weights and export the differences.
    """
    rows = []

    common_years = sorted(set(reference_portfolios.keys()) & set(candidate_portfolios.keys()))

    for Y in common_years:
        ref_info = reference_portfolios[Y]
        cand_info = candidate_portfolios[Y]

        ref_w = pd.Series(ref_info["weights"], index=ref_info["isins"], name="ReferenceWeight")
        cand_w = pd.Series(cand_info["weights"], index=cand_info["isins"], name="CandidateWeight")

        all_isins = sorted(set(ref_w.index) | set(cand_w.index))

        comp = pd.DataFrame(index=all_isins)
        comp["ReferenceWeight"] = ref_w.reindex(all_isins).fillna(0.0)
        comp["CandidateWeight"] = cand_w.reindex(all_isins).fillna(0.0)
        comp["DeltaWeight"] = comp["CandidateWeight"] - comp["ReferenceWeight"]
        comp["AbsDeltaWeight"] = comp["DeltaWeight"].abs()

        conditions = [
            (comp["ReferenceWeight"] == 0) & (comp["CandidateWeight"] > 0),
            (comp["ReferenceWeight"] > 0) & (comp["CandidateWeight"] == 0),
            comp["DeltaWeight"] > 0,
            comp["DeltaWeight"] < 0,
        ]
        labels = ["Added", "Removed", "Overweighted", "Underweighted"]
        comp["ChangeType"] = np.select(conditions, labels, default="Unchanged")

        comp["Year"] = Y + 1
        comp["ISIN"] = comp.index

        rows.append(comp.reset_index(drop=True))

    if rows:
        out = pd.concat(rows, ignore_index=True)
        out = out.sort_values(["Year", "AbsDeltaWeight"], ascending=[True, False])
        out.to_csv(output_csv, index=False)
        print(f"Weight comparison exported: {output_csv}")
    else:
        print(f"Warning: no common years found for {output_csv}")


def export_top_weight_changes(comparison_csv, output_csv, top_n=20):
    """
    From a full weight-comparison CSV, keep only the largest absolute weight changes
    for each year and export them to a separate file.
    """
    df = pd.read_csv(comparison_csv)

    top_df = (
        df.sort_values(["Year", "AbsDeltaWeight"], ascending=[True, False])
          .groupby("Year", group_keys=False)
          .head(top_n)
          .copy()
    )

    top_df.to_csv(output_csv, index=False)
    print(f"Top weight changes exported: {output_csv}")




def ledoit_wolf_cc(X: np.ndarray):
    T, N = X.shape
    S = X.T @ X / T
    var = np.diag(S)
    std = np.sqrt(np.maximum(var, 1e-16))
    std_outer = np.outer(std, std)
    corr = S / std_outer
    np.fill_diagonal(corr, 0.0)
    r_bar = corr.sum() / (N * (N - 1))
    F = r_bar * std_outer
    np.fill_diagonal(F, var)
    X2 = X ** 2
    pi_mat = X2.T @ X2 / T - S ** 2
    pi_hat = pi_mat.sum()
    rho_hat = np.diag(pi_mat).sum()
    X3 = X ** 3
    Theta_II = X3.T @ X / T - S * var[:, None]
    Theta_JJ = X.T @ X3 / T - S * var[None, :]
    ratio_JI = std[None, :] / std[:, None]
    ratio_IJ = std[:, None] / std[None, :]
    rho_mat = (r_bar / 2) * (ratio_JI * Theta_II + ratio_IJ * Theta_JJ)
    rho_hat += rho_mat.sum() - np.diag(rho_mat).sum()
    gamma_hat = np.sum((F - S) ** 2)
    kappa = (pi_hat - rho_hat) / gamma_hat if gamma_hat > 1e-30 else 0.0
    delta = float(np.clip(kappa / T, 0.0, 1.0))
    return delta * F + (1.0 - delta) * S, delta, r_bar


def annual_portfolio_values(monthly_ret: pd.Series, v0: float = 1.0) -> pd.Series:
    values = {Y0: v0}
    v = v0
    for year in range(START_YEAR_OOS, END_YEAR_OOS + 1):
        yr_rets = monthly_ret[monthly_ret.index.year == year]
        if len(yr_rets) > 0:
            v = v * (1.0 + yr_rets).prod()
        values[year] = v
    return pd.Series(values).sort_index()


# ─────────────────────────────────────────────────────────────────────────────


## 4. Data Loading

Load all datasets from the Datastream output files:
- **Static:** Firm-level metadata (ISIN, name, country, region)
- **Monthly RI:** Total return index at month-end (USD)
- **Monthly MV:** Market capitalisation at month-end (M USD)
- **CO₂ Scope 1:** Annual direct emissions (tonnes)
- **Revenue:** Annual revenue (thousands USD → converted to millions)
- **Annual MV:** Year-end market cap (M USD)
- **Risk-free rate:** Monthly (if available)

We filter to firms in the Pacific region and ensure consistency across datasets.

In [34]:
# LOAD COMMON DATA ONCE
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("SAAM Project 2026 — Combined Part I + Part II")
print(f"Region: {REGION_CODE} | Scope: Scope 1")
print("=" * 60)

static = pd.read_excel(data_path("Static_2025.xlsx"), engine="openpyxl")
static.columns = ["ISIN", "NAME", "Country", "Region"]
static["ISIN"] = static["ISIN"].astype(str).str.strip()
pac = static[static["Region"] == REGION_CODE].copy().set_index("ISIN")
pac_isins = set(pac.index)
print(f"Pacific firms in Static: {len(pac_isins)}")

ri_y = load_datastream_wide(data_path("DS_RI_T_USD_Y_2025.xlsx"))
ri_y = ri_y[ri_y.index.isin(pac_isins)].copy()
delist_dates = {isin: extract_delist_date(ri_y.at[isin, "NAME"]) for isin in ri_y.index}
print(f"Delisted Pacific firms: {sum(v is not None for v in delist_dates.values())}")

ri_m_raw = load_datastream_wide(data_path("DS_RI_T_USD_M_2025.xlsx"))
keep_cols, parsed_dates = parse_monthly_columns(list(ri_m_raw.columns))
ri_m = ri_m_raw[["NAME"] + keep_cols].copy()
ri_m.columns = ["NAME"] + parsed_dates
ri_m = ri_m[ri_m.index.isin(pac_isins)].copy()
parsed_dates = [d for d in parsed_dates if d <= pd.Timestamp("2025-12-31")]
ri_m = ri_m[["NAME"] + parsed_dates]
print(f"Monthly RI: {ri_m.shape[0]} firms, {len(parsed_dates)} months")

mv_m_raw = load_datastream_wide(data_path("DS_MV_T_USD_M_2025.xlsx"))
keep_cols_mv, _mv = parse_monthly_columns(list(mv_m_raw.columns))
mv_m = mv_m_raw[["NAME"] + keep_cols_mv].copy()
mv_m.columns = ["NAME"] + _mv
mv_m = mv_m[mv_m.index.isin(pac_isins)].copy()
mv_m = mv_m[["NAME"] + parsed_dates].copy()
mv_m[parsed_dates] = mv_m[parsed_dates].apply(pd.to_numeric, errors="coerce")
print(f"Monthly MV: {mv_m.shape[0]} firms")

common_isins = pac_isins.intersection(set(ri_m.index)).intersection(set(mv_m.index))
ri_m = ri_m.loc[list(common_isins)].copy()
mv_m = mv_m.loc[list(common_isins)].copy()
print(f"Common ISINs: {len(common_isins)}")

ri_prices_universe = clean_monthly_ri_prices(ri_m[parsed_dates], parsed_dates, LOW_PRICE_THRESHOLD, preserve_december_missing=True)
ri_prices_returns = clean_monthly_ri_prices(ri_m[parsed_dates], parsed_dates, LOW_PRICE_THRESHOLD, preserve_december_missing=False)
all_missing = ri_prices_returns.isna().all(axis=1)
if all_missing.any():
    missing_isins = ri_prices_returns.index[all_missing]
    ri_prices_returns = ri_prices_returns.loc[~all_missing].copy()
    ri_prices_universe = ri_prices_universe.loc[~all_missing].copy()
    mv_m = mv_m.drop(index=missing_isins, errors="ignore")
    print(f"Dropped fully-missing RI rows: {all_missing.sum()}")

ri_returns = ri_prices_returns.pct_change(axis=1, fill_method=None)
ri_returns = apply_delisting_to_returns(ri_returns, ri_prices_returns, {k: v for k, v in delist_dates.items() if k in ri_returns.index}, parsed_dates)
ri_returns = apply_terminal_loss_for_permanent_disappearances(ri_returns, ri_prices_returns, parsed_dates)
mv_caps_adjusted = adjust_mv_caps_for_terminal_events(mv_m[parsed_dates], ri_returns, parsed_dates)
print("Monthly returns and adjusted MV caps ready.")

co2_raw = load_annual_panel(data_path(CO2_FILE))
rev_raw = load_annual_panel(data_path("DS_REV_Y_2025.xlsx"))
cap_ann_raw = load_annual_panel(data_path("DS_MV_T_USD_Y_2025.xlsx"))
rev_m = rev_raw / 1000.0  # convert thousands → millions USD

# Forward-fill annual panels: middle/end gaps use previous year (per project rules)
co2_ff = co2_raw.T.ffill().T
co2_panel = co2_ff  # used for investment set filtering (filled = data available)
revM_ff = rev_m.T.ffill().T
capA_ff = cap_ann_raw.T.ffill().T
print(f"CO2 panel: {co2_ff.shape[0]} ISINs, years {co2_ff.columns.min()}–{co2_ff.columns.max()}")

_tmp = pd.read_excel(data_path(CO2_FILE), engine="openpyxl")
_tmp.columns = ["NAME", "ISIN"] + list(_tmp.columns[2:])
_tmp = _tmp[~_tmp["NAME"].astype(str).str.startswith("$$ER", na=False)].dropna(subset=["ISIN"])
_tmp["ISIN"] = _tmp["ISIN"].astype(str).str.strip()
firm_names = _tmp.set_index("ISIN")["NAME"]

try:
    rf_monthly = load_rf_monthly(data_path(RF_FILE))
    print(f"RF loaded: {len(rf_monthly)} obs, mean={rf_monthly.mean() * 100:.3f} %/month")
except Exception as exc:
    print(f"WARNING: could not load RF ({exc}). Using rf=0.")
    rf_monthly = None


def get_carbon_vectors(Y: int, isins: list):
    e = co2_ff[Y].reindex(isins).fillna(0.0).values if Y in co2_ff.columns else np.zeros(len(isins))
    r = revM_ff[Y].reindex(isins).fillna(np.nan).values if Y in revM_ff.columns else np.full(len(isins), np.nan)
    c = capA_ff[Y].reindex(isins).fillna(0.0).values if Y in capA_ff.columns else np.zeros(len(isins))
    return e, r, c



SAAM Project 2026 — Combined Part I + Part II
Region: PAC | Scope: Scope 1
Pacific firms in Static: 513
Delisted Pacific firms: 25
Monthly RI: 513 firms, 313 months
Monthly MV: 513 firms
Common ISINs: 513
Dropped fully-missing RI rows: 1
Monthly returns and adjusted MV caps ready.
CO2 panel: 2545 ISINs, years 1999–2025
RF loaded: 312 obs, mean=0.155 %/month


### 4.1 Data Cleaning Summary

The cleaning pipeline follows the project brief (§1):
1. **Missing prices:** Rows with no data at all are dropped from all tables.
2. **Low prices:** RI values below 0.5 are treated as missing (avoids extreme returns).
3. **Interior missing values:** Forward-filled between two valid observations.
4. **Delisting / bankruptcy:** Return of −100% at the delisting date; subsequent returns set to NaN.
5. **Stale prices:** Firms with >50% zero-return months in the estimation window are excluded.
6. **Carbon filter:** Firms without CO₂ data at year Y are excluded from the investment set.

## 5. Part I — Standard Portfolio Allocation

### 5.1 Investment Set Construction & Minimum-Variance Portfolio

For each decision year Y (2013–2024), we:
1. Define the investment set: Pacific firms with sufficient data and carbon coverage
2. Estimate the pairwise covariance matrix from the trailing 10-year window
3. Solve the long-only minimum-variance problem: $\min_{\alpha} \alpha^\top \Sigma \alpha$ s.t. $\alpha^\top \mathbf{1} = 1$, $\alpha_i \geq 0$
4. Compute monthly OOS portfolio returns with buy-and-hold weight drift

In [35]:

# ─────────────────────────────────────────────────────────────────────────────
# PART I — Standard portfolio allocation
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PART I — Standard Portfolio Allocation")
print("=" * 60)

part1_portfolios = {}
for year_end in range(START_YEAR_OOS - 1, END_YEAR_OOS):
    elig, cols = build_investment_set(ri_prices_universe, ri_returns, parsed_dates, year_end, co2_panel=co2_panel)
    if len(elig) < 2:
        print(f"Year {year_end}: eligible={len(elig)} (skipped)")
        continue
    _, Sigma = estimate_moments(ri_returns, elig, cols)

    if USE_LEDOIT_WOLF:
        R_mat = ri_returns.loc[elig, cols].fillna(0.0).to_numpy().T
        R_dem = R_mat - R_mat.mean(axis=0, keepdims=True)
        Sigma, delta_lw, r_bar_lw = ledoit_wolf_cc(R_dem)
        print(f"  LW shrinkage: δ={delta_lw:.4f}, r̄={r_bar_lw:.4f}")

    w = solve_min_variance(Sigma)
    part1_portfolios[year_end] = {"isins": elig, "weights": w}
    print(f"Year {year_end}: eligible={len(elig)}, max_w={w.max():.4f}")

mv_r = validate_oos_series(compute_mv_oos_returns(part1_portfolios, ri_returns, parsed_dates), "Minimum-variance portfolio")
vw_r = validate_oos_series(compute_vw_oos_returns(part1_portfolios, ri_returns, mv_caps_adjusted, parsed_dates, START_YEAR_OOS, END_YEAR_OOS), "Value-weighted benchmark")

expected_months = pd.date_range(start=f"{START_YEAR_OOS}-01-31", end=f"{END_YEAR_OOS}-12-31", freq="M")
part1_out = pd.DataFrame({"Date": expected_months, "MV_Return": mv_r.values, "VW_Return": vw_r.values})
part1_out["MV_CumReturn"] = (1.0 + part1_out["MV_Return"]).cumprod()
part1_out["VW_CumReturn"] = (1.0 + part1_out["VW_Return"]).cumprod()
part1_out.to_csv(os.path.join(RESULTS_PART1, "part1_results.csv"), index=False)

part1_stats = pd.DataFrame({
    "Minimum Variance": perf_stats(mv_r, rf=rf_monthly),
    "Value Weighted": perf_stats(vw_r, rf=rf_monthly),
}).T
part1_stats.to_csv(os.path.join(RESULTS_PART1, "part1_summary_statistics.csv"))

part1_comp = pd.concat([
    pd.DataFrame({"Year": year_end + 1, "ISIN": info["isins"], "Weight": info["weights"]})
    for year_end, info in part1_portfolios.items()
], ignore_index=True)
part1_comp.to_csv(os.path.join(RESULTS_PART1, "part1_portfolio_compositions.csv"), index=False)

try:
    export_part1_excel_template(
        template_path=data_path("Template for Part I-SAAM.xlsx"),
        output_path=os.path.join(RESULTS_PART1, "Template_for_Part_I_SAAM_FILLED.xlsx"),
        stats=part1_stats,
        out_df=part1_out,
    )
except FileNotFoundError:
    print("Template for Part I not found — skipping Excel export.")

mv_weights_p1 = {yr - 1: grp.set_index("ISIN")["Weight"] for yr, grp in part1_comp.groupby("Year")}
print("Part I outputs written to:", RESULTS_PART1)




PART I — Standard Portfolio Allocation
  LW shrinkage: δ=0.4366, r̄=0.2860
Year 2013: eligible=258, max_w=0.2579
  LW shrinkage: δ=0.4328, r̄=0.2782
Year 2014: eligible=267, max_w=0.2170
  LW shrinkage: δ=0.3934, r̄=0.2794
Year 2015: eligible=288, max_w=0.2387
  LW shrinkage: δ=0.3701, r̄=0.2789
Year 2016: eligible=314, max_w=0.2406
  LW shrinkage: δ=0.3939, r̄=0.2819
Year 2017: eligible=341, max_w=0.2424
  LW shrinkage: δ=0.3748, r̄=0.2551
Year 2018: eligible=365, max_w=0.3925
  LW shrinkage: δ=0.3641, r̄=0.2286
Year 2019: eligible=403, max_w=0.3624
  LW shrinkage: δ=0.3899, r̄=0.2509
Year 2020: eligible=427, max_w=0.2811
  LW shrinkage: δ=0.4528, r̄=0.2444
Year 2021: eligible=457, max_w=0.2726
  LW shrinkage: δ=0.4810, r̄=0.2536
Year 2022: eligible=472, max_w=0.1689
  LW shrinkage: δ=0.4870, r̄=0.2609
Year 2023: eligible=471, max_w=0.1853
  LW shrinkage: δ=0.4693, r̄=0.2561
Year 2024: eligible=469, max_w=0.1894
Part I outputs written to: resultsPart1


### 5.2 Part I — Results

Summary statistics for the minimum-variance portfolio and value-weighted benchmark:

In [36]:
# Display Part I summary statistics
print('Part I — Summary Statistics')
print('=' * 60)
print(part1_stats.round(5).to_string())
print(f'\nOOS months: {len(mv_r)} (MV), {len(vw_r)} (VW)')


Part I — Summary Statistics
                  Annualized Average Return  Annualized Volatility  Annualized Cumulative Return  Sharpe Ratio  Minimum Monthly Return  Maximum Monthly Return
Minimum Variance                    0.07683                0.10825                       0.07339       0.54927                -0.09470                 0.09892
Value Weighted                      0.07369                0.13198                       0.06699       0.42687                -0.11442                 0.13292

OOS months: 144 (MV), 144 (VW)


## 6. Part II — Portfolio Allocation with Carbon Emission Reduction

We now incorporate CO₂ emissions into the portfolio construction. The investment universe
is kept identical to Part I for each year, so that differences arise solely from the carbon constraints.

### 6.1 Carbon Metrics Definitions

- **Carbon Intensity:** $CI_{i,Y} = E_{i,Y} / Revenue_{i,Y}$ (tCO₂/M USD)
- **WACI:** $WACI_Y^{(p)} = \sum_i \alpha_{i,Y} \cdot CI_{i,Y}$
- **Carbon Footprint:** $CF_Y^{(p)} = \sum_i \alpha_{i,Y} \cdot E_{i,Y} / Cap_{i,Y}$

In [37]:

# ─────────────────────────────────────────────────────────────────────────────
# PART II — Carbon objectives and net-zero analysis
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PART II — Portfolio Allocation with Carbon Emission Reduction")
print("=" * 60)

inv_sets_p2 = {}
sigma_p2 = {}
vw_weights_p2 = {}
lw_deltas = {}
lw_r_bars = {}
# Part II uses the SAME eligible universe as Part I for each year,
# so that carbon constraints are the only source of difference.
for Y in DECISION_YEARS:
    if Y not in part1_portfolios:
        print(f"Y={Y}: not available in Part I portfolios — skipping.")
        continue

    # EXACT SAME universe as Part I
    elig = list(part1_portfolios[Y]["isins"])
    cols = window_cols(parsed_dates, Y, WINDOW_YEARS)

    if len(elig) < 2:
        print(f"Y={Y}: only {len(elig)} eligible firms — skipping.")
        continue

    _, Sigma_sample = estimate_moments(ri_returns, elig, cols)

    if USE_LEDOIT_WOLF:
        R_mat = ri_returns.loc[elig, cols].fillna(0.0).to_numpy().T
        R_dem = R_mat - R_mat.mean(axis=0, keepdims=True)
        Sigma, delta_lw, r_bar_lw = ledoit_wolf_cc(R_dem)
        lw_deltas[Y] = delta_lw
        lw_r_bars[Y] = r_bar_lw
        msg = f", δ̂={delta_lw:.4f}, r̄={r_bar_lw:.4f}"
    else:
        Sigma = Sigma_sample
        msg = ""

    sigma_p2[Y] = Sigma

    cap_y = capA_ff[Y].reindex(elig).fillna(0.0).values if Y in capA_ff.columns else np.zeros(len(elig))
    total_cap = cap_y.sum()
    vw_weights_p2[Y] = cap_y / total_cap if total_cap > 0 else np.ones(len(elig)) / len(elig)
    inv_sets_p2[Y] = elig

    print(f"Y={Y}: eligible={len(elig)}{msg}")

vw_weight_dicts = {Y: {"isins": inv_sets_p2[Y], "weights": vw_weights_p2[Y]} for Y in inv_sets_p2}

print("\nSection 3.1 — Carbon intensity, WACI, Carbon Footprint")
rows_31 = {}
for Y in DECISION_YEARS:
    if Y not in inv_sets_p2 or Y not in part1_portfolios:
        continue

    # TRUE Part I MV portfolio
    isins_mv = list(part1_portfolios[Y]["isins"])
    w_mv = part1_portfolios[Y]["weights"]
    e_mv, r_mv, c_mv = get_carbon_vectors(Y, isins_mv)
    ci_mv = carbon_intensity_vec(e_mv, r_mv)

    # VW benchmark on the same eligible universe
    isins_vw = inv_sets_p2[Y]
    w_vw = vw_weights_p2[Y]
    e_vw, r_vw, c_vw = get_carbon_vectors(Y, isins_vw)
    ci_vw = carbon_intensity_vec(e_vw, r_vw)

    rows_31[Y] = {
        "WACI_MV": waci_metric(w_mv, ci_mv),
        "WACI_VW": waci_metric(w_vw, ci_vw),
        "CF_MV": cf_metric(w_mv, e_mv, c_mv),
        "CF_VW": cf_vw_metric(e_vw, c_vw),
    }

df_31 = pd.DataFrame(rows_31).T
df_31.index.name = "Year"
df_31.to_csv(f"{RESULTS_PART2}/carbon_metrics_mv_vw.csv")

waci_contrib_mv, waci_contrib_vw = {}, {}

for Y in DECISION_YEARS:
    if Y not in inv_sets_p2 or Y not in part1_portfolios:
        continue

    # MV contributors from the true Part I MV portfolio
    isins_mv = list(part1_portfolios[Y]["isins"])
    w_mv = part1_portfolios[Y]["weights"]
    e_mv, r_mv, _ = get_carbon_vectors(Y, isins_mv)
    ci_mv = carbon_intensity_vec(e_mv, r_mv)
    contrib_mv = pd.Series(np.nan_to_num(w_mv * ci_mv, nan=0.0), index=isins_mv)

    # VW contributors from benchmark weights
    isins_vw = inv_sets_p2[Y]
    w_vw = vw_weights_p2[Y]
    e_vw, r_vw, _ = get_carbon_vectors(Y, isins_vw)
    ci_vw = carbon_intensity_vec(e_vw, r_vw)
    contrib_vw = pd.Series(np.nan_to_num(w_vw * ci_vw, nan=0.0), index=isins_vw)

    for isin, val in contrib_mv.items():
        waci_contrib_mv.setdefault(isin, []).append(val)

    for isin, val in contrib_vw.items():
        waci_contrib_vw.setdefault(isin, []).append(val)

top10_mv = pd.Series({k: np.mean(v) for k, v in waci_contrib_mv.items()}).sort_values(ascending=False).head(10)
top10_vw = pd.Series({k: np.mean(v) for k, v in waci_contrib_vw.items()}).sort_values(ascending=False).head(10)
top10_mv_df = pd.DataFrame({"ISIN": top10_mv.index, "Name": top10_mv.index.map(firm_names), "Avg_WACI_Contribution_MV": top10_mv.values})
top10_vw_df = pd.DataFrame({"ISIN": top10_vw.index, "Name": top10_vw.index.map(firm_names), "Avg_WACI_Contribution_VW": top10_vw.values})
top10_mv_df.to_csv(f"{RESULTS_PART2}/top10_waci_contributors_mv.csv", index=False)
top10_vw_df.to_csv(f"{RESULTS_PART2}/top10_waci_contributors_vw.csv", index=False)




PART II — Portfolio Allocation with Carbon Emission Reduction
Y=2013: eligible=258, δ̂=0.4366, r̄=0.2860
Y=2014: eligible=267, δ̂=0.4328, r̄=0.2782
Y=2015: eligible=288, δ̂=0.3934, r̄=0.2794
Y=2016: eligible=314, δ̂=0.3701, r̄=0.2789
Y=2017: eligible=341, δ̂=0.3939, r̄=0.2819
Y=2018: eligible=365, δ̂=0.3748, r̄=0.2551
Y=2019: eligible=403, δ̂=0.3641, r̄=0.2286
Y=2020: eligible=427, δ̂=0.3899, r̄=0.2509
Y=2021: eligible=457, δ̂=0.4528, r̄=0.2444
Y=2022: eligible=472, δ̂=0.4810, r̄=0.2536
Y=2023: eligible=471, δ̂=0.4870, r̄=0.2609
Y=2024: eligible=469, δ̂=0.4693, r̄=0.2561

Section 3.1 — Carbon intensity, WACI, Carbon Footprint


### 6.2 Carbon Footprint & WACI — Unconstrained Portfolios

The table below shows the annual carbon footprint (CF) and weighted-average carbon intensity (WACI)
for both the minimum-variance portfolio and the value-weighted benchmark.

In [38]:
# Display carbon metrics
print('Carbon Metrics — MV vs VW')
print(df_31.round(2).to_string())
print('\nTop 10 WACI contributors — MV portfolio:')
print(top10_mv_df.to_string(index=False))
print('\nTop 10 WACI contributors — VW benchmark:')
print(top10_vw_df.to_string(index=False))


Carbon Metrics — MV vs VW
      WACI_MV  WACI_VW   CF_MV   CF_VW
Year                                  
2013  2436.35   191.51  682.28  227.93
2014  2046.95   175.88  677.97  209.15
2015  5552.73   346.24  560.39  217.54
2016  6542.15   443.46  560.13  265.12
2017  5277.78   517.29  502.34  232.77
2018  3616.02   395.54  710.59  181.35
2019  1239.25   300.37  532.32  194.76
2020  1196.96   242.44  567.62  155.04
2021  1705.06   201.54  659.10  151.63
2022   941.40   228.04  392.09  152.75
2023   605.06   205.51  338.17  155.03
2024   566.82   204.75  370.59  147.76

Top 10 WACI contributors — MV portfolio:
        ISIN                    Name  Avg_WACI_Contribution_MV
HK0006000050   POWER ASSETS HOLDINGS               1649.739801
HK0002007356            CLP HOLDINGS                853.040292
BMG2178K1009            CKI HOLDINGS                119.335122
JP3429800000            ANA HOLDINGS                 12.587331
JP3180400008               OSAKA GAS                 10.628357
MU0117U0

### 6.3 Section 3.2 — Minimum-Variance Portfolio with 50% Carbon Footprint Cap

We solve: $\min_{\alpha} \alpha^\top \Sigma \alpha$ subject to $CF^{(p)}_Y \leq 0.5 \times CF^{(P^{mv}_{oos})}_Y$, $\alpha^\top \mathbf{1} = 1$, $\alpha_i \geq 0$.

This portfolio halves the carbon footprint of the unconstrained minimum-variance allocation.

In [41]:
print("\nSection 3.2 — Min-variance with 50% Carbon Footprint constraint")
port_32, cf_32, waci_32 = {}, {}, {}
for Y in DECISION_YEARS:
    if Y not in inv_sets_p2:
        continue
    isins = inv_sets_p2[Y]
    Sigma = sigma_p2[Y]
    e, r, c = get_carbon_vectors(Y, isins)
    ec = e_over_c_vec(e, c)
    cf_mv_Y = df_31.loc[Y, "CF_MV"]
    if np.isnan(cf_mv_Y) or cf_mv_Y <= 0:
        print(f"Y={Y}: CF_MV invalid — skipping 3.2.")
        continue
    cf_target = 0.5 * cf_mv_Y
    w = solve_cvxpy(Sigma, ec, cf_target, mode="mv")
    if w is None:
        raise ValueError(f"Section 3.2 | Y={Y}: optimizer failed. Do not fallback silently.")
    achieved_cf = validate_cf_constraint(w, e, c, cf_target, "Section 3.2", Y)

    w_mv_Y = np.array(part1_portfolios[Y]["weights"])  # ✅ Part I MV weights for year Y
    cf_unconstrained = cf_metric(w_mv_Y, e, c)
    print(f"Y={Y}: CF_MV={cf_unconstrained:.4f}, CF_target={cf_target:.4f}, "
      f"CF_achieved={achieved_cf:.4f}, binding={abs(achieved_cf - cf_target) < 1e-4}")

    port_32[Y] = {"isins": isins, "weights": w}
    cf_32[Y] = achieved_cf
    waci_32[Y] = waci_metric(w, carbon_intensity_vec(e, r))

if port_32:
    pd.concat([
        pd.DataFrame({"Year": Y, "ISIN": v["isins"], "Weight": v["weights"]})
        for Y, v in port_32.items()
    ], ignore_index=True).to_csv(
        f"{RESULTS_PART2}/weights_32_mv_carbon05.csv",
        index=False
    )
else:
    print("Warning: port_32 is empty. No weights_32_mv_carbon05.csv file created.")

ret_32 = validate_oos_series(
    compute_oos_returns(port_32, ri_returns, parsed_dates, START_YEAR_OOS, END_YEAR_OOS),
    "P_mv_oos(0.5)"
)
ret_32.to_csv(f"{RESULTS_PART2}/returns_32_mv_carbon05.csv", header=["Return"])
stats_32 = pd.DataFrame({"P_mv_oos": perf_stats_extended(mv_r, rf=rf_monthly), "P_mv_oos(0.5)": perf_stats_extended(ret_32, rf=rf_monthly)}).T
stats_32.to_csv(f"{RESULTS_PART2}/stats_32_comparison.csv")




Section 3.2 — Min-variance with 50% Carbon Footprint constraint
Y=2013: CF_MV=682.2787, CF_target=341.1394, CF_achieved=341.1394, binding=True
Y=2014: CF_MV=677.9662, CF_target=338.9831, CF_achieved=338.9831, binding=True
Y=2015: CF_MV=560.3887, CF_target=280.1944, CF_achieved=280.1920, binding=False
Y=2016: CF_MV=560.1284, CF_target=280.0642, CF_achieved=280.0642, binding=True
Y=2017: CF_MV=502.3397, CF_target=251.1699, CF_achieved=251.1699, binding=True
Y=2018: CF_MV=710.5853, CF_target=355.2926, CF_achieved=355.2926, binding=True
Y=2019: CF_MV=532.3172, CF_target=266.1586, CF_achieved=266.1586, binding=True
Y=2020: CF_MV=567.6208, CF_target=283.8104, CF_achieved=283.8104, binding=True
Y=2021: CF_MV=659.1005, CF_target=329.5502, CF_achieved=329.5502, binding=True
Y=2022: CF_MV=392.0891, CF_target=196.0446, CF_achieved=196.0446, binding=True
Y=2023: CF_MV=338.1726, CF_target=169.0863, CF_achieved=169.0863, binding=True
Y=2024: CF_MV=370.5860, CF_target=185.2930, CF_achieved=185.2930,

### 6.4 Section 3.3 — Tracking-Error Minimisation with 50% Carbon Cap

We solve: $\min_{\alpha} (\alpha - \alpha^{vw})^\top \Sigma (\alpha - \alpha^{vw})$ subject to
$CF^{(p)}_Y \leq 0.5 \times CF^{(P^{vw})}_Y$, $\alpha^\top \mathbf{1} = 1$, $\alpha_i \geq 0$.

This "otherwise passive" strategy stays close to the benchmark while cutting emissions by half.

In [42]:
print("\nSection 3.3 — TE minimisation with 50% Carbon Footprint constraint")
port_33, cf_33, waci_33 = {}, {}, {}
for Y in DECISION_YEARS:
    if Y not in inv_sets_p2:
        continue
    isins = inv_sets_p2[Y]
    Sigma = sigma_p2[Y]
    e, r, c = get_carbon_vectors(Y, isins)
    ec = e_over_c_vec(e, c)
    w_vw = vw_weights_p2[Y]
    cf_vw_Y = df_31.loc[Y, "CF_VW"]
    if np.isnan(cf_vw_Y) or cf_vw_Y <= 0:
        print(f"Y={Y}: CF_VW invalid — skipping 3.3.")
        continue
    cf_target = 0.5 * cf_vw_Y
    w = solve_cvxpy(Sigma, ec, cf_target, w_ref=w_vw, mode="te")
    if w is None:
        raise ValueError(f"Section 3.3 | Y={Y}: optimizer failed. Do not fallback silently.")
    achieved_cf = validate_cf_constraint(w, e, c, cf_target, "Section 3.3", Y)

    print(f"Y={Y}: CF_VW={cf_vw_Y:.4f}, CF_target={cf_target:.4f}, "
      f"CF_achieved={achieved_cf:.4f}, binding={abs(achieved_cf - cf_target) < 1e-4}")
      
    port_33[Y] = {"isins": isins, "weights": w}
    cf_33[Y] = achieved_cf
    waci_33[Y] = waci_metric(w, carbon_intensity_vec(e, r))

if port_33:
    pd.concat([
        pd.DataFrame({"Year": Y, "ISIN": v["isins"], "Weight": v["weights"]})
        for Y, v in port_33.items()
    ], ignore_index=True).to_csv(
        f"{RESULTS_PART2}/weights_33_te_carbon05.csv",
        index=False
    )
else:
    print("Warning: port_33 is empty. No weights_33_te_carbon05.csv file created.")

ret_33 = validate_oos_series(
    compute_oos_returns(port_33, ri_returns, parsed_dates, START_YEAR_OOS, END_YEAR_OOS),
    "P_vw_oos(0.5)"
)
ret_33.to_csv(f"{RESULTS_PART2}/returns_33_te_carbon05.csv", header=["Return"])
stats_33 = pd.DataFrame({"P_vw_oos": perf_stats_extended(vw_r, rf=rf_monthly), "P_vw_oos(0.5)": perf_stats_extended(ret_33, rf=rf_monthly)}).T
stats_33.to_csv(f"{RESULTS_PART2}/stats_33_comparison.csv")




Section 3.3 — TE minimisation with 50% Carbon Footprint constraint
Y=2013: CF_VW=227.9316, CF_target=113.9658, CF_achieved=113.9658, binding=True
Y=2014: CF_VW=209.1464, CF_target=104.5732, CF_achieved=104.5731, binding=False
Y=2015: CF_VW=217.5363, CF_target=108.7681, CF_achieved=108.7681, binding=True
Y=2016: CF_VW=265.1164, CF_target=132.5582, CF_achieved=132.5534, binding=False
Y=2017: CF_VW=232.7717, CF_target=116.3858, CF_achieved=116.3858, binding=True
Y=2018: CF_VW=181.3526, CF_target=90.6763, CF_achieved=90.6760, binding=False
Y=2019: CF_VW=194.7648, CF_target=97.3824, CF_achieved=97.3824, binding=True
Y=2020: CF_VW=155.0398, CF_target=77.5199, CF_achieved=77.5199, binding=True
Y=2021: CF_VW=151.6313, CF_target=75.8156, CF_achieved=75.8155, binding=False
Y=2022: CF_VW=152.7499, CF_target=76.3750, CF_achieved=76.3750, binding=True
Y=2023: CF_VW=155.0271, CF_target=77.5135, CF_achieved=77.5135, binding=True
Y=2024: CF_VW=147.7589, CF_target=73.8795, CF_achieved=73.8795, binding

### 6.5 Section 4.1 — Net-Zero Portfolio (10% Annual Decarbonisation)

We implement a dynamic decarbonisation trajectory: $CF^{(p)}_Y \leq (1 - \theta)^{Y - Y_0 + 1} \times CF^{(P^{vw})}_{Y_0}$
with $\theta = 10\%$ and $Y_0 = 2013$.

This forces financed emissions to decline by at least 10% per year from the 2013 benchmark level.

In [43]:
print(f"\nSection 4.1 — Net-zero portfolio (θ={THETA:.0%}/yr from Y0={Y0})")
isins_y0 = inv_sets_p2.get(Y0, [])
if isins_y0:
    e0, _, c0 = get_carbon_vectors(Y0, isins_y0)
    cf_vw_y0 = cf_vw_metric(e0, c0)
else:
    cf_vw_y0 = df_31.loc[Y0, "CF_VW"] if Y0 in df_31.index else np.nan
print(f"Fixed anchor CF_VW at Y0={Y0}: {cf_vw_y0:.4f}")   
port_41, cf_41, waci_41 = {}, {}, {}


for Y in DECISION_YEARS:
    if Y not in inv_sets_p2:
        continue
    isins = inv_sets_p2[Y]
    Sigma = sigma_p2[Y]
    e, r, c = get_carbon_vectors(Y, isins)
    ec = e_over_c_vec(e, c)
    w_vw = vw_weights_p2[Y]
    cf_target = ((1.0 - THETA) ** (Y - Y0)) * cf_vw_y0
    print(f"Y={Y}: cf_target_nz={cf_target:.4f}  "
          f"(cumulative reduction={(1-(1-THETA)**(Y-Y0))*100:.1f}%)")
    w = solve_cvxpy(Sigma, ec, cf_target, w_ref=w_vw, mode="te")
    if w is None:
        raise ValueError(f"Section 4.1 | Y={Y}: optimizer failed. Do not fallback silently.")
    achieved_cf = validate_cf_constraint(w, e, c, cf_target, "Section 4.1", Y)

    print(f"Y={Y}: CF_VW_anchor=227.93, CF_target={cf_target:.4f}, "
      f"CF_achieved={achieved_cf:.4f}, binding={abs(achieved_cf - cf_target) < 1e-4}")
      
    port_41[Y] = {"isins": isins, "weights": w}
    cf_41[Y] = achieved_cf
    waci_41[Y] = waci_metric(w, carbon_intensity_vec(e, r))

if port_41:
    pd.concat([
        pd.DataFrame({"Year": Y, "ISIN": v["isins"], "Weight": v["weights"]})
        for Y, v in port_41.items()
    ], ignore_index=True).to_csv(
        f"{RESULTS_PART2}/weights_41_netzero.csv",
        index=False
    )
else:
    print("Warning: port_41 is empty. No weights_41_netzero.csv file created.")

ret_41 = validate_oos_series(
    compute_oos_returns(port_41, ri_returns, parsed_dates, START_YEAR_OOS, END_YEAR_OOS),
    "P_vw_oos(NZ)"
)
ret_41.to_csv(f"{RESULTS_PART2}/returns_41_netzero.csv", header=["Return"])
stats_41 = pd.DataFrame({
    "P_vw_oos": perf_stats_extended(vw_r, rf=rf_monthly),
    "P_vw_oos(0.5)": perf_stats_extended(ret_33, rf=rf_monthly),
    "P_vw_oos(NZ)": perf_stats_extended(ret_41, rf=rf_monthly),
}).T
stats_41.to_csv(f"{RESULTS_PART2}/stats_41_comparison.csv")




Section 4.1 — Net-zero portfolio (θ=10%/yr from Y0=2013)
Fixed anchor CF_VW at Y0=2013: 227.9316
Y=2013: cf_target_nz=227.9316  (cumulative reduction=0.0%)
Y=2013: CF_VW_anchor=227.93, CF_target=227.9316, CF_achieved=227.9316, binding=True
Y=2014: cf_target_nz=205.1384  (cumulative reduction=10.0%)
Y=2014: CF_VW_anchor=227.93, CF_target=205.1384, CF_achieved=205.1384, binding=True
Y=2015: cf_target_nz=184.6246  (cumulative reduction=19.0%)
Y=2015: CF_VW_anchor=227.93, CF_target=184.6246, CF_achieved=184.6246, binding=True
Y=2016: cf_target_nz=166.1621  (cumulative reduction=27.1%)
Y=2016: CF_VW_anchor=227.93, CF_target=166.1621, CF_achieved=166.1621, binding=True
Y=2017: cf_target_nz=149.5459  (cumulative reduction=34.4%)
Y=2017: CF_VW_anchor=227.93, CF_target=149.5459, CF_achieved=149.5459, binding=True
Y=2018: cf_target_nz=134.5913  (cumulative reduction=41.0%)
Y=2018: CF_VW_anchor=227.93, CF_target=134.5913, CF_achieved=134.5913, binding=True
Y=2019: cf_target_nz=121.1322  (cumulat

## 7. Comparison & Synthesis

### 7.1 Portfolio Values and Attributed CO₂

Assuming $V_{2013} = 1$ M USD starting wealth, we track how portfolio value
evolves and compute the total attributed CO₂ emissions each year.

In [44]:
print("\nTotal attributed CO2 (V_2013 = $1 M starting wealth)…")
V0_M = 1.0
v_mv = annual_portfolio_values(mv_r, V0_M)
v_vw = annual_portfolio_values(vw_r, V0_M)
v_32 = annual_portfolio_values(ret_32, V0_M)
v_33 = annual_portfolio_values(ret_33, V0_M)
v_41 = annual_portfolio_values(ret_41, V0_M)
cf_mv_s = df_31["CF_MV"]
cf_vw_s = df_31["CF_VW"]
cf_32_s = pd.Series(cf_32).sort_index()
cf_33_s = pd.Series(cf_33).sort_index()
cf_41_s = pd.Series(cf_41).sort_index()

total_co2 = pd.DataFrame({
    "TotalCO2_MV": v_mv.reindex(df_31.index) * cf_mv_s,
    "TotalCO2_VW": v_vw.reindex(df_31.index) * cf_vw_s,
    "TotalCO2_MV05": v_32.reindex(pd.Index(sorted(cf_32_s.index))) * cf_32_s,
    "TotalCO2_TE05": v_33.reindex(pd.Index(sorted(cf_33_s.index))) * cf_33_s,
    "TotalCO2_NZ": v_41.reindex(pd.Index(sorted(cf_41_s.index))) * cf_41_s,
    "V_MV_MUSD": v_mv.reindex(df_31.index),
    "V_VW_MUSD": v_vw.reindex(df_31.index),
    "V_MV05_MUSD": v_32.reindex(pd.Index(sorted(cf_32_s.index))),
    "V_TE05_MUSD": v_33.reindex(pd.Index(sorted(cf_33_s.index))),
    "V_NZ_MUSD": v_41.reindex(pd.Index(sorted(cf_41_s.index))),
}).round(4)
total_co2.index.name = "Year"
total_co2.to_csv(f"{RESULTS_PART2}/total_attributed_co2.csv")

all_stats = pd.DataFrame({
    "P_mv_oos": perf_stats_extended(mv_r, rf=rf_monthly),
    "P_mv_oos(0.5)": perf_stats_extended(ret_32, rf=rf_monthly),
    "P_vw_oos": perf_stats_extended(vw_r, rf=rf_monthly),
    "P_vw_oos(0.5)": perf_stats_extended(ret_33, rf=rf_monthly),
    "P_vw_oos(NZ)": perf_stats_extended(ret_41, rf=rf_monthly),
}).T
all_stats.to_csv(f"{RESULTS_PART2}/all_portfolio_stats_extended.csv")

rel_stats = pd.DataFrame({
    "P_vw_oos(0.5)": perf_stats_relative(ret_33, vw_r, port_33, vw_weight_dicts, rf=rf_monthly),
    "P_vw_oos(NZ)": perf_stats_relative(ret_41, vw_r, port_41, vw_weight_dicts, rf=rf_monthly),
}).T
rel_stats.to_csv(f"{RESULTS_PART2}/relative_stats_te_portfolios.csv")

carbon_df = pd.DataFrame({
    "CF_MV": cf_mv_s,
    "CF_VW": cf_vw_s,
    "CF_MV_05": cf_32_s,
    "CF_TE_05": cf_33_s,
    "CF_NZ": cf_41_s,
    "WACI_MV": df_31["WACI_MV"],
    "WACI_VW": df_31["WACI_VW"],
    "WACI_MV_05": pd.Series(waci_32),
    "WACI_TE_05": pd.Series(waci_33),
    "WACI_NZ": pd.Series(waci_41),
})
if USE_LEDOIT_WOLF:
    carbon_df["LW_delta"] = pd.Series(lw_deltas)
    carbon_df["LW_r_bar"] = pd.Series(lw_r_bars)
carbon_df.rename_axis("Year").to_csv(f"{RESULTS_PART2}/all_carbon_metrics.csv")

nz_path = pd.Series({Y: ((1.0 - THETA) ** (Y - Y0 + 1)) * cf_vw_y0 for Y in DECISION_YEARS}, name="NZ_target").rename_axis("Year")
nz_path.to_csv(f"{RESULTS_PART2}/nz_target_path.csv")

pd.DataFrame({
    "P_mv_oos": perf_stats_extended(mv_r, rf=rf_monthly),
    "P_mv_oos(0.5)": perf_stats_extended(ret_32, rf=rf_monthly),
    "P_vw_oos": perf_stats_extended(vw_r, rf=rf_monthly),
    "P_vw_oos(0.5)": perf_stats_extended(ret_33, rf=rf_monthly),
}).T.to_csv(f"{RESULTS_PART2}/stats_34_section_comparison.csv")

pd.DataFrame({
    "P_vw_oos": perf_stats_extended(vw_r, rf=rf_monthly),
    "P_vw_oos(0.5)": perf_stats_extended(ret_33, rf=rf_monthly),
    "P_vw_oos(NZ)": perf_stats_extended(ret_41, rf=rf_monthly),
}).T.to_csv(f"{RESULTS_PART2}/stats_42_section_comparison.csv")




Total attributed CO2 (V_2013 = $1 M starting wealth)…


### 7.2 Summary Statistics — All Five Portfolios

Comparison of annualised return, volatility, Sharpe ratio, drawdown, and carbon metrics:

In [45]:
# Display the five-portfolio comparison
print('All Portfolio Statistics')
print('=' * 80)
print(all_stats.round(5).to_string())
print('\nRelative Statistics (TE portfolios vs VW benchmark):')
print(rel_stats.round(5).to_string())
print('\nCarbon Metrics Summary:')
print(carbon_df.round(2).to_string())


All Portfolio Statistics
               Annualized Average Return  Annualized Volatility  Annualized Cumulative Return  Sharpe Ratio  Minimum Monthly Return  Maximum Monthly Return  VaR_95 (monthly)  VaR_99 (monthly)  ES_95 (monthly)  ES_99 (monthly)  Max_Drawdown
P_mv_oos                         0.07683                0.10825                       0.07339       0.54927                -0.09470                 0.09892           0.04169           0.06681          0.05987          0.08149      -0.17475
P_mv_oos(0.5)                    0.07689                0.11269                       0.07294       0.52819                -0.10474                 0.10388           0.04348           0.06408          0.05963          0.08629      -0.17392
P_vw_oos                         0.07369                0.13198                       0.06699       0.42687                -0.11442                 0.13292           0.06626           0.09361          0.08520          0.10753      -0.22852
P_vw_oos(0.5)  

### 7.3 Figures

#### Cumulative Returns & Drawdowns

In [46]:
save_cumret_plot({"P_mv_oos": mv_r, "P_mv_oos(0.5)": ret_32}, f"{RESULTS_PART2}/cumret_32_mv_vs_mv05.png", "Section 3.2 — Cumulative Returns")
save_cumret_plot({"P_vw_oos": vw_r, "P_vw_oos(0.5)": ret_33}, f"{RESULTS_PART2}/cumret_33_vw_vs_te05.png", "Section 3.3 — Cumulative Returns")
save_cumret_plot({"P_vw_oos": vw_r, "P_vw_oos(0.5)": ret_33, "P_vw_oos(NZ)": ret_41}, f"{RESULTS_PART2}/cumret_41_42_vw_te05_nz.png", "Section 4.1 / 4.2 — Cumulative Returns")

# --- Enhanced plots: cumulative + drawdown ---
plot_cumret_drawdown(
    {"P(mv)_oos": mv_r, "P(vw)": vw_r},
    f"{RESULTS_PART2}/plot_cumret_dd_mv_vw.png",
    "Minimum-Variance vs Value-Weighted Benchmark"
)
plot_cumret_drawdown(
    {"P(vw)": vw_r, "P(mv)_oos": mv_r, "P(mv)_oos(0.5)": ret_32},
    f"{RESULTS_PART2}/plot_cumret_dd_section32.png",
    "Section 3.2 — MV Portfolio with 50% Carbon Cap"
)
plot_cumret_drawdown(
    {"P(vw)": vw_r, "P(mv)_oos": mv_r, "P(mv)_oos(0.5)": ret_32, "P(vw)_oos(0.5)": ret_33},
    f"{RESULTS_PART2}/plot_cumret_dd_section33.png",
    "Section 3.3 — Tracking-Error Portfolio with 50% Carbon Cap"
)
plot_cumret_drawdown(
    {"P(vw)": vw_r, "P(mv)_oos": mv_r, "P(mv)_oos(0.5)": ret_32,
     "P(vw)_oos(0.5)": ret_33, "P(vw)_oos(NZ)": ret_41},
    f"{RESULTS_PART2}/plot_cumret_dd_all5.png",
    "All Five Portfolio Strategies — Cumulative Returns & Drawdowns"
)

# --- Enhanced plots: CF bar charts ---
plot_cf_bar_comparison(
    {"CF_MV": cf_mv_s.to_dict(), "CF_VW": cf_vw_s.to_dict()},
    f"{RESULTS_PART2}/plot_cf_bars_mv_vw.png",
    "Carbon Footprint — MV vs VW Benchmark"
)
# CF with 50% constraint line
mv_cf_target = {Y: 0.5 * cf_mv_s.get(Y, np.nan) for Y in cf_mv_s.index}
plot_cf_bar_comparison(
    {"CF_MV": cf_mv_s.to_dict(), "CF_MV(0.5)": cf_32_s.to_dict()},
    f"{RESULTS_PART2}/plot_cf_bars_section32.png",
    "Section 3.2 — MV Carbon Footprint with 50% Cap",
    constraint_dict={"50% of CF_MV": mv_cf_target}
)
vw_cf_target = {Y: 0.5 * cf_vw_s.get(Y, np.nan) for Y in cf_vw_s.index}
plot_cf_bar_comparison(
    {"CF_VW": cf_vw_s.to_dict(), "CF_TE(0.5)": cf_33_s.to_dict()},
    f"{RESULTS_PART2}/plot_cf_bars_section33.png",
    "Section 3.3 — TE Portfolio CF with 50% VW Cap",
    constraint_dict={"50% of CF_VW": vw_cf_target}
)
nz_targets = {Y: ((1.0 - THETA) ** (Y - Y0 + 1)) * cf_vw_y0 for Y in cf_41_s.index}
plot_cf_bar_comparison(
    {"CF_VW": cf_vw_s.to_dict(), "CF_NZ": cf_41_s.to_dict()},
    f"{RESULTS_PART2}/plot_cf_bars_netzero.png",
    "Section 4.1 — Net-Zero CF Trajectory",
    constraint_dict={"NZ 10% p.a. target": nz_targets}
)

# --- Enhanced plots: WACI bar charts ---
plot_cf_bar_comparison(
    {"WACI_MV": df_31["WACI_MV"].to_dict(), "WACI_VW": df_31["WACI_VW"].to_dict()},
    f"{RESULTS_PART2}/plot_waci_bars_mv_vw.png",
    "Weighted-Average Carbon Intensity — MV vs VW"
)
plot_cf_bar_comparison(
    {"WACI_MV": df_31["WACI_MV"].to_dict(), "WACI_MV(0.5)": waci_32},
    f"{RESULTS_PART2}/plot_waci_bars_section32.png",
    "Section 3.2 — WACI: MV vs MV(0.5)"
)
plot_cf_bar_comparison(
    {"WACI_VW": df_31["WACI_VW"].to_dict(), "WACI_TE(0.5)": waci_33, "WACI_NZ": waci_41},
    f"{RESULTS_PART2}/plot_waci_bars_te_nz.png",
    "Sections 3.3 & 4.1 — WACI: VW vs TE(0.5) vs NZ"
)

# --- Correlation matrix of monthly returns ---


#### Correlation Matrix of Monthly Returns

In [47]:
print("\nCorrelation matrix of monthly portfolio returns:")
ret_all = pd.DataFrame({
    "P(mv)_oos": mv_r,
    "P(mv)_oos(0.5)": ret_32,
    "P(vw)": vw_r,
    "P(vw)_oos(0.5)": ret_33,
    "P(vw)_oos(NZ)": ret_41,
}).dropna()
corr_matrix = ret_all.corr().values
corr_labels = list(ret_all.columns)
print(ret_all.corr().round(4).to_string())
ret_all.corr().to_csv(f"{RESULTS_PART2}/correlation_matrix.csv")
plot_correlation_heatmap(corr_matrix, corr_labels, f"{RESULTS_PART2}/plot_correlation_heatmap.png")




Correlation matrix of monthly portfolio returns:
                P(mv)_oos  P(mv)_oos(0.5)   P(vw)  P(vw)_oos(0.5)  P(vw)_oos(NZ)
P(mv)_oos          1.0000          0.9867  0.6704          0.6674         0.6683
P(mv)_oos(0.5)     0.9867          1.0000  0.6755          0.6723         0.6730
P(vw)              0.6704          0.6755  1.0000          0.9974         0.9974
P(vw)_oos(0.5)     0.6674          0.6723  0.9974          1.0000         1.0000
P(vw)_oos(NZ)      0.6683          0.6730  0.9974          1.0000         1.0000


#### Carbon Footprint & WACI Evolution

In [48]:
# keep the simple line plots too (useful for quick checks)
cf_plot_df = pd.DataFrame({
    "CF_MV": cf_mv_s,
    "CF_MV_05": cf_32_s,
    "CF_VW": cf_vw_s,
    "CF_TE_05": cf_33_s,
    "CF_NZ": cf_41_s,
}).sort_index()
save_annual_line_plot(cf_plot_df, f"{RESULTS_PART2}/carbon_footprint_paths.png", "Carbon Footprint by Year", "Carbon Footprint (tCO2 / M USD invested)")

waci_plot_df = pd.DataFrame({
    "WACI_MV": df_31["WACI_MV"],
    "WACI_MV_05": pd.Series(waci_32),
    "WACI_VW": df_31["WACI_VW"],
    "WACI_TE_05": pd.Series(waci_33),
    "WACI_NZ": pd.Series(waci_41),
}).sort_index()
save_annual_line_plot(waci_plot_df, f"{RESULTS_PART2}/waci_paths.png", "WACI by Year", "WACI (tCO2 / M USD revenue)")



### 7.4 Weight Composition Analysis

Compare portfolio weights between constrained and unconstrained portfolios to identify
which firms are over/underweighted or excluded by the carbon constraints.

In [49]:
if port_32:
    export_weight_comparison(
        reference_portfolios=part1_portfolios,
        candidate_portfolios=port_32,
        output_csv=f"{RESULTS_PART2}/weights_comparison_32_vs_part1_mv.csv"
    )
    export_top_weight_changes(
        f"{RESULTS_PART2}/weights_comparison_32_vs_part1_mv.csv",
        f"{RESULTS_PART2}/top_weight_changes_32_vs_part1_mv.csv",
        top_n=20
    )
else:
    print("Warning: port_32 is empty. No comparison files created for Section 3.2.")

if port_33:
    export_weight_comparison(
        reference_portfolios=vw_weight_dicts,
        candidate_portfolios=port_33,
        output_csv=f"{RESULTS_PART2}/weights_comparison_33_vs_vw.csv"
    )
    export_top_weight_changes(
        f"{RESULTS_PART2}/weights_comparison_33_vs_vw.csv",
        f"{RESULTS_PART2}/top_weight_changes_33_vs_vw.csv",
        top_n=20
    )
else:
    print("Warning: port_33 is empty. No comparison files created for Section 3.3.")

if port_41:
    export_weight_comparison(
        reference_portfolios=vw_weight_dicts,
        candidate_portfolios=port_41,
        output_csv=f"{RESULTS_PART2}/weights_comparison_41_vs_vw.csv"
    )
    export_top_weight_changes(
        f"{RESULTS_PART2}/weights_comparison_41_vs_vw.csv",
        f"{RESULTS_PART2}/top_weight_changes_41_vs_vw.csv",
        top_n=20
    )
else:
    print("Warning: port_41 is empty. No comparison files created for Section 4.1.")

print("\nDone. Results written to:")
print("  ", RESULTS_PART1)
print("  ", RESULTS_PART2)




Weight comparison exported: ResultsPart2_FINAL/weights_comparison_32_vs_part1_mv.csv
Top weight changes exported: ResultsPart2_FINAL/top_weight_changes_32_vs_part1_mv.csv
Weight comparison exported: ResultsPart2_FINAL/weights_comparison_33_vs_vw.csv
Top weight changes exported: ResultsPart2_FINAL/top_weight_changes_33_vs_vw.csv
Weight comparison exported: ResultsPart2_FINAL/weights_comparison_41_vs_vw.csv
Top weight changes exported: ResultsPart2_FINAL/top_weight_changes_41_vs_vw.csv

Done. Results written to:
   resultsPart1
   ResultsPart2_FINAL


In [50]:
pd.DataFrame({"CF_target_NZ": cf_41, "CF_achieved_NZ": cf_41}).T
# cf_41[Y] should be ≤ cf_target for all Y

,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
CF_target_NZ,227.931581,205.138423,184.624581,166.162123,149.545911,134.59132,121.132188,109.018969,98.117073,88.305365,79.474797,71.527345
CF_achieved_NZ,227.931581,205.138423,184.624581,166.162123,149.545911,134.59132,121.132188,109.018969,98.117073,88.305365,79.474797,71.527345


In [51]:
print("=" * 75)
print("CONSTRAINT BINDING DIAGNOSTIC — Sections 3.2, 3.3, 4.1")
print("=" * 75)

sections = {
    "3.2 — MV + 50% CF_MV":   (port_32, cf_32, "MV"),
    "3.3 — TE + 50% CF_VW":   (port_33, cf_33, "TE_05"),
    "4.1 — TE + Net Zero":     (port_41, cf_41, "NZ"),
}

targets_32 = {Y: 0.5 * df_31.loc[Y, "CF_MV"] for Y in port_32 if Y in df_31.index}
targets_33 = {Y: 0.5 * df_31.loc[Y, "CF_VW"] for Y in port_33 if Y in df_31.index}
targets_41 = {Y: ((1.0 - THETA) ** (Y - Y0)) * cf_vw_y0 for Y in port_41}

targets = {"3.2 — MV + 50% CF_MV": targets_32,
           "3.3 — TE + 50% CF_VW": targets_33,
           "4.1 — TE + Net Zero":   targets_41}

for section_name, (port, cf_achieved_dict, _) in sections.items():
    print(f"\n{section_name}")
    print(f"{'Year':<6} {'CF_target':>12} {'CF_achieved':>13} {'Slack':>12} {'Binding':>9} {'Status'}")
    print("-" * 70)
    all_ok = True
    for Y in sorted(port.keys()):
        target   = targets[section_name][Y]
        achieved = cf_achieved_dict[Y]
        slack    = target - achieved          # positive = constraint not fully tight
        binding  = abs(slack) < 1e-4
        violated = achieved > target + 1e-4
        if violated:
            status = "❌ VIOLATED"
            all_ok = False
        elif binding:
            status = "✅ binding"
        else:
            status = "○  not binding (slack)"
        print(f"{Y:<6} {target:>12.4f} {achieved:>13.4f} {slack:>12.6f} {str(binding):>9} {status}")
    print(f"  → {'All constraints OK ✅' if all_ok else 'WARNING: violation detected ❌'}")

print("\n" + "=" * 75)
print("INTERPRETATION")
print("=" * 75)
print("binding=True  : optimizer pushed exactly to constraint boundary (expected)")
print("not binding   : constraint is loose — VW already satisfies target naturally")
print("               (expected for NZ early years 2013-2015)")
print("VIOLATED      : BUG — achieved CF exceeds target beyond tolerance")

CONSTRAINT BINDING DIAGNOSTIC — Sections 3.2, 3.3, 4.1

3.2 — MV + 50% CF_MV
Year      CF_target   CF_achieved        Slack   Binding Status
----------------------------------------------------------------------
2013       341.1394      341.1394     0.000000      True ✅ binding
2014       338.9831      338.9831     0.000001      True ✅ binding
2015       280.1944      280.1920     0.002367     False ○  not binding (slack)
2016       280.0642      280.0642     0.000003      True ✅ binding
2017       251.1699      251.1699     0.000005      True ✅ binding
2018       355.2926      355.2926     0.000000      True ✅ binding
2019       266.1586      266.1586     0.000000      True ✅ binding
2020       283.8104      283.8104     0.000000      True ✅ binding
2021       329.5502      329.5502     0.000000      True ✅ binding
2022       196.0446      196.0446     0.000000      True ✅ binding
2023       169.0863      169.0863     0.000000      True ✅ binding
2024       185.2930      185.2930     

## Use of Large Language Models (LLMs)

This project used Claude (Anthropic) as a support tool for:
- **Debugging:** Identifying and fixing errors in data cleaning and optimisation logic
- **Code structure:** Improving the organisation and readability of the codebase
- **Plotting:** Generating professional-quality matplotlib figures
- **Report polishing:** Improving clarity and grammar of written sections

All core methodological choices (investment set construction, covariance estimation approach,
optimisation problem formulations, carbon metric definitions, and interpretation of results)
are the group's own work. The group is fully responsible for the correctness and academic
integrity of the submission.